# Bronze → Silver Layer Migration

This notebook performs the data exploration and migration from the Bronze layer (`01-bronze`) to the Silver layer (`02-silver`). 

The Silver layer will contain cleaned, typed, and deduplicated tables.

Silver Design strategy approache: Kinball approach  
- Botton-up: Business process driven instead of enterprise
- Do not apply 3 Normal Form and do not consider SCD7 (slowly changing dimension type 7): 
    - Speed vs Data consistency with high integrity
    - High data redundancy vs Highly normalized data applying 3NF
    - Decision: Keep PK as hash-based value


## General Bronze Data Exploration Analysis

| `geolocation` | 1,000,163 | 280K duplicate rows, lat/lng as STRING |
| `order_items` | 112,650 | shipping_limit_date as STRING |
| `order_payments` | 103,886 | Clean — minor standardization needed |
| `order_reviews` | 104,162 | review_score as STRING with invalid values (dates, Portuguese text), 1,205 duplicate review_ids |
| `orders` | 99,441 | All date columns stored as STRING |
| `product_category_name_translation` | 71 | Clean |
| `products` | 32,951 | 610 null categories, column name typos (`lenght`) |
| `sellers` | 3,095 | Clean |

### Some general transformations for this layer decisions:

- **Column pruning**: Dropping all null columns (_rescued_data) and bronze metadata columns (source_file, ingestion_time)
- **Nullif**: Null when a string value is empty after trimming
- **Type casting**: Convert datetime string columns into TIMESTAMP type, lat/lng strings → `DOUBLE`, review_score string -> `INT`
- **Data cleaning**: Filter invalid review_scores (keep 1–5), fix column name typos (`lenght` -> `length`)
- **Deduplication**: Remove duplicate `review_id` and geolocation rows
- **Standardization**: 
    - TRIM text fields
    - UPPER state codes, status
    - Ttable name in singular instead of plural used on Bronze
- **Metadata**: Add `record_ingestion_timestamp` column to all Silver tables

## Specific Data Exploration Analysis 
Each table contains a DEA session that will explore the bronze ingested data to understand how the data has been ingested and assess the design and migration to silver layer



In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;
SET TIME ZONE 'America/Sao_Paulo';


---

## Customers Ingestion

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;

-- Explore bronze customers: schema overview + duplicate customer_id check

-- Quick look at the table structure and sample rows
DESCRIBE `01-bronze`.customers;

SELECT * FROM `01-bronze`.customers LIMIT 10;

-- Find duplicates in customer table by customer_id (should be zero — PK)
SELECT customer_id, COUNT(*) AS occurrence_count
FROM `01-bronze`.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking for fully duplicate rows in orders
SELECT COUNT(*) AS full_duplicate_count
FROM (
  SELECT *
  FROM `01-bronze`.customers
  GROUP BY ALL
  HAVING COUNT(*) > 1
);

-- Find duplicates in customer table by customer_unique_id (expected — one customer can have multiple customer_ids)
SELECT
  customer_unique_id,
  COUNT(*) AS occurrence_count
FROM `01-bronze`.customers
GROUP BY customer_unique_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking some of the duplicated customer_unique_ids
SELECt c.customer_unique_id, c.customer_id
FROM `01-bronze`.customers c
WHERE customer_unique_id = '8d50f5eadf50201ccdcedfb9e2ac8455';

-- Checking the relation with order table to see if there are multiple orders for the same customer_id
SELECT o.order_id, o.customer_id, c.customer_unique_id
FROM `01-bronze`.orders o, `01-bronze`.customers c
WHERE o.customer_id = c.customer_id AND c.customer_unique_id = 'd44ccec15f5f86d14d6a2cfa67da1975';


-- Check all distinct values for customer_state in bronze customers
SELECT DISTINCT customer_state
FROM `01-bronze`.customers
ORDER BY customer_state;

-- Checking what has changed on customer data for the customer that has changed its customer_id
WITH customer_duplicated AS (
    SELECT
        customer_unique_id,
        COUNT(*) AS occurrence_count
    FROM `01-bronze`.customers
    WHERE customer_unique_id IS NOT NULL
    GROUP BY customer_unique_id
    HAVING COUNT(*) > 1
),  customer_data_grouped AS (  
    SELECT
    c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
    FROM `01-bronze`.customers c
    JOIN customer_duplicated cd
    ON c.customer_unique_id = cd.customer_unique_id
    GROUP BY c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
), customer_disticted_data AS (
    SELECT 
        cg.customer_unique_id, count(*) occurrence_count
    FROM customer_data_grouped cg
    GROUP BY cg.customer_unique_id
    HAVING occurrence_count > 1
    ORDER BY occurrence_count DESC
)
SELECT  c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix, cdd.occurrence_count
FROM customer_disticted_data cdd, customer_data_grouped c
WHERE cdd.customer_unique_id = c.customer_unique_id
ORDER BY cdd.occurrence_count, cdd.customer_unique_id  DESC;

-- Checking the results of the above query in customer table direclty.  
SELECT DISTINCT c.customer_unique_id, c.customer_city, c.customer_state, c.customer_zip_code_prefix
FROM `01-bronze`.customers c 
WHERE c.customer_unique_id in ('3e43e6105506432c953e165fb2acf44c', 'd44ccec15f5f86d14d6a2cfa67da1975')
ORDER BY c.customer_unique_id;




## Customer Data Analysis

### Considerations
- No duplicate customer_id has been found - No need to deduplicate on this id
- No duplicate rows in customers table - No need to deduplicate
- No invalid state has been found (27 total - including fedeal district)
- Customer duplicated ids (cutomer_unique_id and customer_id)
    - Analisys to understand why 2 id's are available and what is the relationship with other tables
    - It has been identified that customer_id is the customer key to orders table. Each order refers to a single customer_id
    - After running the above SQL's its possible to check that, for some customer_unique_id, there are multiple customer_id. 
    - See picture below that shows different customer data for an unique customer_unique_id.
- It has been identified that the original data uses a hash value as a key (cutomer_unique_id and customer_id), must probably generated by a MD5 algorithm. 
    - When migrating to Silver layer, it has been considered if a creation of a INT/BIGINT surrogate key is a good approach
    - A reason to consider changing the key's to int is performance on joins. Joins using string have a worse performance when comparing with numeric id's
    - For the pourpouse of this project, the ammount of data is not significative to impact performance
    - Spring joins does not have a significant worse performance in Apache Spark, as discussed on [this topic](https://discord.com/channels/1521498663011221615/1521498664608989200/1552793656484696125). 
    - Therefore, after a deep research, and using [this discution](https://community.databricks.com/t5/warehousing-analytics/guid-or-concatenated-string-as-a-primary-key-in-silver-and-gold/m-p/150204/highlight/true#M2524) as reference, in order to garantee indepotency when recreating Silver layer, it has been decided to keep to keep the hash id comming from bronze layer
 

    !['customer duplicated data'](customer_id_duplicated_data_reason.png)

    - The reason for this... best guess... is to preserv the historical data of a customer (as customer is a mutable entity)
    - decision: keep as it is. No change on silver

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;

-- Table customer — select native columns, trim text, standardize state. 
-- Adding a _record_created_timestamp column for auditing purposes
-- Customer City, Zipcode and State column will not be normalized using 3NF as per Inmon approach
-- Keeping the original Primary key (customer_id) as hash based value

CREATE OR REPLACE TABLE customer AS
SELECT
  TRIM(customer_id) AS customer_id,
  TRIM(customer_unique_id) AS customer_unique_id,
  cast(NULLIF(TRIM(customer_zip_code_prefix), '') AS int)AS customer_zip_code_prefix,
  NULLIF(TRIM(customer_city), '') AS customer_city,
  NULLIF(UPPER(TRIM(customer_state)), '') AS customer_state,
  current_timestamp() AS record_ingestion_timestamp
FROM `01-bronze`.customers
WHERE customer_id IS NOT NULL;

-- Set customer_id as NOT NULL (CTAS defaults all columns to nullable)
ALTER TABLE customer ALTER COLUMN customer_id SET NOT NULL;

-- Set customer_id as NOT NULL (CTAS defaults all columns to nullable)
ALTER TABLE customer ALTER COLUMN customer_unique_id SET NOT NULL;

-- Define customer_id as the primary key of the silver customers table
ALTER TABLE customer
ADD CONSTRAINT pk_customer PRIMARY KEY (customer_id);

-- Row Count Reconciliation
SELECT 'customer' AS silver_table_name,
  (SELECT COUNT(*) FROM `01-bronze`.customers) AS bronze_count,
  (SELECT COUNT(*) FROM customer) AS silver_count,
  silver_count - bronze_count AS silver_minus_bronze,
  CASE
    WHEN (silver_count) = (bronze_count) THEN 'MATCH'
    WHEN (silver_count) > (bronze_count) THEN 'SILVER_HIGHER'
    ELSE 'BRONZE_HIGHER'
  END AS reconciliation_status
;




---

## Orders Ingestion

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;

DESCRIBE TABLE `01-bronze`.orders;

-- Checking for duplicate records in orders by order_id
SELECT order_id, COUNT(*) AS occurrence_count
FROM `01-bronze`.orders
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking for fully duplicate rows in orders
SELECT COUNT(*) AS full_duplicate_count
FROM (
  SELECT *
  FROM `01-bronze`.orders
  GROUP BY ALL
  HAVING COUNT(*) > 1
);

-- Checking for nullable customer_id, order_id in orders, order_status, order_purchase_timestamp
SELECT count(*) as customer_id_null_count
FROM `01-bronze`.orders
WHERE TRIM(customer_id) = '' or customer_id IS NULL;

SELECT count(*) as order_id_null_count
FROM `01-bronze`.orders
WHERE TRIM(order_id) = '' or order_id IS NULL;

SELECT count(*) as order_status_null_count
FROM `01-bronze`.orders
WHERE TRIM(order_status) = '' or order_status IS NULL;

SELECT count(*) as order_purchase_timestamp_null_count
FROM `01-bronze`.orders
WHERE TRIM(order_purchase_timestamp) = '' or order_purchase_timestamp IS NULL;

-- checking if a customer_id in orders table is not present in customers table
SELECT customer_id
FROM `01-bronze`.orders
EXCEPT
SELECT customer_id
FROM customer;

-- checking dates and timestamp values for correct typecasting and timezone conversion
SELECT order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date, order_purchase_timestamp
FROM `01-bronze`.orders
LIMIT 5;


-- Verifing typecast using timezone conversion
SET TIME ZONE 'America/Sao_Paulo';
SELECT 
    TRY_CAST(order_approved_at as timestamp) AS timestamp_br, order_approved_at
FROM `01-bronze`.orders
LIMIT 5;

-- Check all distinct values for order_status in bronze orders
SELECT DISTINCT order_status
FROM `01-bronze`.orders
ORDER BY order_status;




### Orders - Bronze Layer Data Analysis
#### Considerations
- No duplicate order_id has been found - No need to deduplicate on this id
- order_id as primary key
- customer_id as a foreing key from customer table
    - Interesting to check that an order is related to the customer_id instead of customer_unique_id
    - The reason for this seems to be related to the customer mutable address aspect from customer _entity_
- No duplicate rows in orders table - No need to deduplicate
- Date/datetime columns: 

    - The date/datetime columns needs conversion to use timezone
    - Pattern on bronze layer does not contain timezone information
    - Assuming Sao Paulo timezone
    - It has been considered to cast to TIMESTAMP_NTZ type. There are some [limitations](https://docs.databricks.com/aws/en/sql/language-manual/data-types/timestamp-ntz-type#notes) - prefered to not use it
    - order_delivered_customer_date is the only one that is a date and not datetime
- Order status 
    - No null available in data. Not null 
    - Set of well defined values 

    ![Order Status Values](order_status.png)




In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;
SET TIME ZONE 'America/Sao_Paulo';

-- create table order from bronze
CREATE OR REPLACE TABLE order AS
SELECT
  TRIM(order_id) AS order_id,
  TRIM(customer_id) AS customer_id,
  NULLIF(UPPER(TRIM(order_status)), '') AS order_status,
  TRY_CAST(NULLIF(order_purchase_timestamp, '') AS TIMESTAMP) AS order_purchase_timestamp,
  TRY_CAST(NULLIF(order_approved_at, '') AS TIMESTAMP) AS order_approved_at,
  TRY_CAST(NULLIF(order_delivered_carrier_date, '') AS TIMESTAMP) AS order_delivered_carrier_date,
  TRY_CAST(NULLIF(order_delivered_customer_date, '') AS DATE) AS order_delivered_customer_date,
  TRY_CAST(NULLIF(order_estimated_delivery_date, '') AS TIMESTAMP) AS order_estimated_delivery_date,
  current_timestamp() AS record_ingestion_timestamp
FROM `01-bronze`.orders
WHERE order_id IS NOT NULL;

ALTER TABLE order ALTER COLUMN customer_id SET NOT NULL;

ALTER TABLE order ALTER COLUMN order_id SET NOT NULL;

ALTER TABLE order ALTER COLUMN order_status SET NOT NULL;

ALTER TABLE order ALTER COLUMN order_purchase_timestamp SET NOT NULL;

ALTER TABLE order
ADD CONSTRAINT pk_order PRIMARY KEY (order_id);

-- Foreign key: order.customer_id -> customer.customer_id
ALTER TABLE order
ADD CONSTRAINT fk_order_customer FOREIGN KEY (customer_id)
REFERENCES customer (customer_id);

-- Row Count Reconciliation
SELECT 'order' AS silver_table_name,
  (SELECT COUNT(*) FROM `01-bronze`.orders) AS bronze_count,
  (SELECT COUNT(*) FROM order) AS silver_count,
  silver_count - bronze_count AS silver_minus_bronze,
  CASE
    WHEN (silver_count) = (bronze_count) THEN 'MATCH'
    WHEN (silver_count) > (bronze_count) THEN 'SILVER_HIGHER'
    ELSE 'BRONZE_HIGHER'
  END AS reconciliation_status
;

---

## Products Ingestion

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;

DESCRIBE TABLE `01-bronze`.products;

SELECT * FROM `01-bronze`.products
LIMIT 5;

-- Checking for duplicate records in products by product_id
SELECT product_id, COUNT(*) AS occurrence_count
FROM `01-bronze`.products
GROUP BY product_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking for fully duplicate rows in products
SELECT COUNT(*) AS full_duplicate_count
FROM (
  SELECT *
  FROM `01-bronze`.products
  GROUP BY ALL
  HAVING COUNT(*) > 1
);

-- Checking for nullable product_id
SELECT COUNT(*) AS product_id_null_count
FROM `01-bronze`.products
WHERE TRIM(product_id) = '' OR product_id IS NULL;

-- Check distinct product_category_name values
SELECT DISTINCT product_category_name
FROM `01-bronze`.products
ORDER BY product_category_name;


-- Check if all products contain a category (null or empty product_category_name)
SELECT COUNT(*) AS products_missing_category_count
FROM `01-bronze`.products
WHERE product_category_name IS NULL OR TRIM(product_category_name) = '';



### Products - Bronze Layer Data Analysis
#### Considerations
- No duplicate product_id has been found - No need to deduplicate on this id
- product_id as primary key
- No duplicate rows in products table - No need to deduplicate
- Products table does not contain product name or descriptions. Instead, contains product name lenght and product description lenght and some other product attributes
    - Considered if these product attributes are needed and decided that some analysis can be answered by these columns like "Does the product name size or product weight have any influence on product sale numbers?"
    - Decided to keep it in Silver layer for now
- Product Category - product_category_name column
    - product_category_name uses Brazilian Portuguese language. A translation table is available in Bronze layer but it want be used for now as this project goal is to BR use. To do for a next phase
    - A question to be answered in gold layer is related to sales per categories along periods. For this reason the list of categories in products table will be normalized for better search and BI dashboards use
    - The relation between category and product will be: A product is from 1 and only 1 category and a category can have 0 or many products
    - 73 distinct categories + 'null' category in product_category_name column
    - 610 products are not related to any category (null value). Creating a new category called 'unknown' to assign to these products
    - Some category names seems to be duplicated although with different names (ex. eletrodomesticos and eletrodomesticos_2)
    - Some categogy values in Bronze layer:
    
    ![](categories.png) 


### Product and Category tables creation

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;
    
-------------------- Category -------------------------------
-- Create table category with distinct product_category_name values and a category_id as a surrogate pk
CREATE OR REPLACE TABLE category AS
SELECT
  ROW_NUMBER() OVER (ORDER BY product_category_name) AS category_id,
  product_category_name
FROM (
  SELECT DISTINCT COALESCE(TRIM(product_category_name), 'unknown') AS product_category_name
  FROM `01-bronze`.products
  ORDER BY product_category_name
)
WHERE product_category_name IS NOT NULL;

ALTER TABLE category ALTER COLUMN category_id SET NOT NULL;

ALTER TABLE category ALTER COLUMN product_category_name SET NOT NULL;

ALTER TABLE category
ADD CONSTRAINT pk_category PRIMARY KEY (category_id);

-------------------- Product  -------------------------------

-- Create table product from bronze layer, joining category to get category_id
CREATE OR REPLACE TABLE product AS
SELECT
  TRIM(p.product_id) AS product_id,
  c.category_id,
  -- COALESCE(TRIM(p.product_category_name), 'unknown') AS product_category_name,
  p.product_name_lenght AS product_name_length,
  p.product_description_lenght AS product_description_length,
  p.product_photos_qty,
  p.product_weight_g,
  p.product_length_cm,
  p.product_height_cm,
  p.product_width_cm,
  current_timestamp() AS record_ingestion_timestamp
FROM `01-bronze`.products p
LEFT JOIN category c
  ON COALESCE(TRIM(p.product_category_name), 'unknown') = c.product_category_name
WHERE p.product_id IS NOT NULL;

ALTER TABLE product ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE product ALTER COLUMN category_id SET NOT NULL;

ALTER TABLE product
ADD CONSTRAINT pk_product PRIMARY KEY (product_id);

-- Foreign key: product.category_id -> category.category_id
ALTER TABLE product
ADD CONSTRAINT fk_product_category FOREIGN KEY (category_id)
REFERENCES category (category_id);

-------------------- Reconciliation -------------------------------
-- Row Count Reconciliation for category
SELECT 'category' AS silver_table_name,
  (SELECT COUNT(DISTINCT (TRIM(product_category_name), 'unknown'))
   FROM `01-bronze`.products) AS bronze_count,
  (SELECT COUNT(*) FROM category) AS silver_count,
  silver_count - bronze_count AS silver_minus_bronze,
  CASE
    WHEN (silver_count) = (bronze_count) THEN 'MATCH'
    WHEN (silver_count) > (bronze_count) THEN 'SILVER_HIGHER'
    ELSE 'BRONZE_HIGHER'
  END AS reconciliation_status
;



-- Row Count Reconciliation for product
SELECT 'product' AS silver_table_name,
  (SELECT COUNT(*) FROM `01-bronze`.products) AS bronze_count,
  (SELECT COUNT(*) FROM product) AS silver_count,
  silver_count - bronze_count AS silver_minus_bronze,
  CASE
    WHEN (silver_count) = (bronze_count) THEN 'MATCH'
    WHEN (silver_count) > (bronze_count) THEN 'SILVER_HIGHER'
    ELSE 'BRONZE_HIGHER'
  END AS reconciliation_status
;

-- Check every product.category_id exists in category table
SELECT COUNT(*) AS orphan_category_id_count
FROM (
  SELECT category_id FROM product
  EXCEPT
  SELECT category_id FROM category
);





---

## Order Items Ingestion

In [0]:
%sql

USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;

-- Quick look at the table structure
DESCRIBE TABLE `01-bronze`.order_items;

-- Sample rows to understand the data
SELECT * FROM `01-bronze`.order_items LIMIT 10;

-- Checking for duplicate records in order_items by order_id + order_item_id (composite PK candidate)
SELECT order_id, order_item_id, COUNT(*) AS occurrence_count
FROM `01-bronze`.order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

-- Checking for fully duplicate rows in order_items
SELECT COUNT(*) AS full_duplicate_count
FROM (
  SELECT *
  FROM `01-bronze`.order_items
  GROUP BY ALL
  HAVING COUNT(*) > 1
);

-- Checking for nullable / empty order_id
SELECT COUNT(*) AS order_id_null_count
FROM `01-bronze`.order_items
WHERE TRIM(order_id) = '' OR order_id IS NULL;

-- Checking for nullable / empty product_id
SELECT COUNT(*) AS product_id_null_count
FROM `01-bronze`.order_items
WHERE TRIM(product_id) = '' OR product_id IS NULL;


-- Checking for null or non-positive price and freight_value
SELECT COUNT(*) AS price_null_or_zero_count
FROM `01-bronze`.order_items
WHERE price IS NULL OR price <= 0;

SELECT COUNT(*) AS freight_value_null_count
FROM `01-bronze`.order_items
WHERE freight_value IS NULL;

-- Checking for null or empty shipping_limit_date
SELECT COUNT(*) AS shipping_limit_date_null_count
FROM `01-bronze`.order_items
WHERE TRIM(shipping_limit_date) = '' OR shipping_limit_date IS NULL;

-- Verify typecasting for shipping_limit_date using timezone conversion
SET TIME ZONE 'America/Sao_Paulo';
SELECT
  TRY_CAST(NULLIF(shipping_limit_date, '') AS TIMESTAMP) AS shipping_limit_timestamp_br,
  shipping_limit_date
FROM `01-bronze`.order_items
LIMIT 5;

-- Check if order_id in order_items exists in orders table 
SELECT COUNT(*) AS orphan_order_id_count
FROM (
  SELECT order_id
  FROM `01-bronze`.order_items
  EXCEPT
  SELECT order_id
  FROM `01-bronze`.orders
);

-- Check if product_id in order_items exists in products table 
SELECT COUNT(*) AS orphan_product_id_count
FROM (
  SELECT product_id
  FROM `01-bronze`.order_items
  EXCEPT
  SELECT product_id
  FROM `01-bronze`.products
);



-- Check order_item_id distribution (should be sequential per order starting at 1)
SELECT order_item_id, COUNT(*) AS occurrence_count
FROM `01-bronze`.order_items
GROUP BY order_item_id
ORDER BY order_item_id;



 ### Order Items - Bronze Layer Data Analysis

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;
USE SCHEMA `02-silver`;
SET TIME ZONE 'America/Sao_Paulo';

-- Silver: order_items — cast shipping_limit_date, keep native columns
CREATE OR REPLACE TABLE `02-silver`.order_item AS
SELECT
  TRIM(order_id) AS order_id,
  order_item_id,
  TRIM(product_id) AS product_id,
  TRIM(seller_id) AS seller_id,
  TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
  price,
  freight_value,
  current_timestamp() AS silver_ingestion_time
FROM `01-bronze`.order_items
WHERE order_id IS NOT NULL;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;

-- Silver: order_payments — standardize payment_type, keep native columns
CREATE OR REPLACE TABLE `02-silver`.order_payments AS
SELECT
  TRIM(order_id) AS order_id,
  payment_sequential,
  LOWER(TRIM(payment_type)) AS payment_type,
  payment_installments,
  payment_value,
  current_timestamp() AS silver_ingestion_time
FROM `01-bronze`.order_payments
WHERE order_id IS NOT NULL;

In [0]:
%sql
USE CATALOG puc_data_specialist_de_2026_09;

-- Silver: order_reviews — cast review_score to INT (filter invalid), deduplicate, cast dates to TIMESTAMP
CREATE OR REPLACE TABLE `02-silver`.order_reviews AS
SELECT
  TRIM(review_id) AS review_id,
  TRIM(order_id) AS order_id,
  CAST(review_score AS INT) AS review_score,
  TRIM(review_comment_title) AS review_comment_title,
  TRIM(review_comment_message) AS review_comment_message,
  TRY_CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
  TRY_CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
  current_timestamp() AS silver_ingestion_time
FROM (
  SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp,
    ROW_NUMBER() OVER (
      PARTITION BY review_id
      ORDER BY review_creation_date DESC
    ) AS rn
  FROM `01-bronze`.order_reviews
  WHERE review_id IS NOT NULL
    AND review_score IN ('1', '2', '3', '4', '5')
)
WHERE rn = 1;